# Etapa 5 — Avaliação de Desempenho

**Objetivo:** Medir objetivamente qual modelo é melhor e por quê.

Um modelo que acerta 90% pode parecer ótimo — mas se a classe 3 (Golfadas Severas)
só tem 2% dos dados, um modelo que ignora essa classe completamente já acerta 98%!
Por isso usamos métricas mais criteriosas:

- **F1-macro:** penaliza quando o modelo ignora classes raras
- **Matriz de confusão:** mostra exatamente quais classes estão sendo confundidas
- **Análise por tipo de fonte:** um modelo bom em dados simulados mas ruim em dados reais não serve para campo

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_predict

from config import FEATURES_DATA_PATH, MODELS_DIR, N_SPLITS_CV
from src.evaluation import (
    compare_models,
    compute_metrics,
    evaluate_by_source,
    plot_confusion_matrix,
    print_classification_report,
)

print('Bibliotecas carregadas!')

## 5.1 Carregar dados e modelos

In [ ]:
df = pd.read_parquet(FEATURES_DATA_PATH)

META_COLS = ['instance_id', 'fault_class', 'source_type', 'window_start']
feature_cols = [c for c in df.columns if c not in META_COLS]

X = df[feature_cols].values
y = df['fault_class'].values
groups = df['instance_id'].values
source_types = df['source_type'].values

imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Carregar modelos salvos
model_files = sorted(MODELS_DIR.glob('*.joblib'))
models = {f.stem: joblib.load(f) for f in model_files}
print(f'Modelos carregados: {list(models.keys())}')

## 5.2 Predições por cross-validation

Usamos `cross_val_predict` para gerar predições em todo o conjunto usando GroupKFold.
Cada janela é predita pelo modelo que nunca a viu no treino — avaliação honesta.

In [ ]:
gkf = GroupKFold(n_splits=N_SPLITS_CV)
all_metrics = []
predictions = {}

for model_name, model in models.items():
    print(f'\nAvaliando {model_name}...')

    y_pred = cross_val_predict(model, X, y, cv=gkf, groups=groups, n_jobs=-1)
    predictions[model_name] = y_pred

    # Tentar obter probabilidades (nem todos os modelos suportam)
    try:
        y_proba = cross_val_predict(model, X, y, cv=gkf, groups=groups,
                                     method='predict_proba', n_jobs=-1)
    except Exception:
        y_proba = None

    metrics = compute_metrics(y, y_pred, y_proba, model_name=model_name)
    all_metrics.append(metrics)
    print(f'  F1-macro: {metrics["f1_macro"]:.4f} | Accuracy: {metrics["accuracy"]:.4f}')

print('\nPredições geradas para todos os modelos!')

## 5.3 Tabela comparativa

In [ ]:
df_comparison = compare_models(all_metrics)
plt.show()

## 5.4 Relatório detalhado por classe — melhor modelo

In [ ]:
best_model_name = df_comparison.index[0]  # maior F1-macro
print(f'Melhor modelo: {best_model_name}')

print_classification_report(y, predictions[best_model_name], model_name=best_model_name)

## 5.5 Matrizes de confusão — todos os modelos

In [ ]:
for model_name, y_pred in predictions.items():
    fig = plot_confusion_matrix(y, y_pred, model_name=model_name)
    plt.show()

## 5.6 Desempenho por tipo de fonte

Esta análise é crítica para o TCC: o modelo precisa funcionar em dados reais (WELL),
não apenas em dados simulados.

In [ ]:
for model_name, y_pred in predictions.items():
    evaluate_by_source(y, y_pred, source_types, model_name=model_name)